Now As part of silver first we will fetch the customer_raw from Schema bronze and lets see what modifications are needed 

In [0]:
%sql
select * from workspace.bronze.customers_raw

fetching how many duplicate customer do we have 

In [0]:
%sql
select customer_id, count(*) as count_customer from workspace.bronze.customers_raw group by customer_id having count_customer>1;

In [0]:
%sql
select * from workspace.bronze.customers_raw where customer_id='C003'

Analysing the each row and what we can do 

In [0]:
from pyspark.sql import functions as F
customer_df=spark.table("workspace.bronze.customers_raw")
customer_cleaning_df=customer_df.dropDuplicates(["customer_id"])
display(customer_cleaning_df.filter(F.col("state").isNull()))
display(customer_cleaning_df.filter(F.col("city").isNull()))
display(customer_cleaning_df.filter((F.col("state") == "Karnataka") | (F.col("state") == "Kerala")))
display(customer_cleaning_df.filter(F.col("customer_id").isNull()))


Now we will remove null customer_id directly 

In [0]:
#by using the below isNotNull we will keep all not null rows and remove nulls 
customer_cleaning_df = customer_cleaning_df.filter(F.col("customer_id").isNotNull())
display(customer_cleaning_df)

now we will replace customer name and city from Null to unknown

In [0]:
customer_cleaning_df = customer_cleaning_df.fillna("Unknown", subset=["customer_name", "city"])
display(customer_cleaning_df)

Here we are making states from Null to exact values from comparing with existing records

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define window partitioned by city to find the known state for each city
window_city = Window.partitionBy("city")

# 2. Infer missing state using existing values for the same city, then fill remaining NULLs with "Unknown"
# ✅ Correct way:
customer_cleaning_df = customer_cleaning_df \
    .withColumn("state", F.coalesce(F.col("state"), F.max("state").over(window_city))) \
    .fillna("Unknown", subset=["city", "state"])

display(customer_cleaning_df)

Here we have trimmed all the columns and sorted the data frame completly with customer id

In [0]:
from pyspark.sql import functions as F
customer_cleaning_df=customer_cleaning_df.withColumn("customer_id",F.trim(F.col("customer_id")))
customer_cleaning_df=customer_cleaning_df.withColumn("customer_name",F.trim(F.col("customer_name")))
customer_cleaning_df=customer_cleaning_df.withColumn("city",F.trim(F.col("city")))
customer_cleaning_df=customer_cleaning_df.withColumn("state",F.trim(F.col("state")))
customer_cleaning_df=customer_cleaning_df.withColumn("ingested_timestamp",F.trim(F.col("ingested_timestamp")))
customer_cleaning_df=customer_cleaning_df.sort("customer_id")
display(customer_cleaning_df)

Loading the final customer_cleaning_df in to silver schema with table name customers_silver 


In [0]:
customer_cleaning_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.customer_silver")
display(customer_cleaning_df)

In [0]:
%sql
select* from workspace.silver.customer_silver

Now we will start looking in to orders_raw and its cleansing 

In [0]:
order_cleaning_df=spark.table("workspace.bronze.orders_raw")
display(order_cleaning_df)

In [0]:
#now we will remove excat duplaicates of each rows 
order_cleaning_df=order_cleaning_df.dropDuplicates()
display(order_cleaning_df)

In [0]:
#finding same orderid with different customer_id
from pyspark.sql import functions as F
order_cleaning_Order_find=order_cleaning_df.groupBy("order_id").agg(F.count("customer_id").alias("cust_Order_count")).filter(F.col("cust_Order_count")>1)
display(order_cleaning_Order_find)


In [0]:
from pyspark.sql import functions as F
order_cleaning_filter=order_cleaning_df.select("order_id","customer_id","product_id","order_date","quantity","ingested_timestamp").filter(F.col("order_id")=="O1003")
display(order_cleaning_filter)
order_cleaning_filter.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.order_id_exception")

In [0]:
from pyspark.sql import functions as F
order_cleaning_df=order_cleaning_df.filter(F.col("order_id")!="O1003")
display(order_cleaning_df)

In [0]:
from pyspark.sql import functions as F
order_cleaning_null=order_cleaning_df.filter(F.col("customer_id").isNull())
display(order_cleaning_null)
order_cleaning_null.write.format("delta").mode("append").saveAsTable("workspace.silver.order_id_exception")

In [0]:
%sql
select * from workspace.silver.order_id_exception

In [0]:
from pyspark.sql import functions as F
#we have removed the customer_id null
order_cleaning_df=order_cleaning_df.filter(F.col("customer_id").isNotNull())
display(order_cleaning_df)

Here we will filter the informat date and future dates 

In [0]:
from pyspark.sql import functions as F

# Filter rows where try_to_date returns NULL (invalid format) OR date is in the future
invalid_date_rows_df = order_cleaning_df.filter(
    F.try_to_date(F.col("order_date"), "yyyy-MM-dd").isNull() | 
    (F.try_to_date(F.col("order_date"), "yyyy-MM-dd") > F.current_date())
)

display(invalid_date_rows_df)
invalid_date_rows_df.write.format("delta").mode("append").saveAsTable("workspace.silver.order_id_exception")

In [0]:
from pyspark.sql import functions as F

# Keep only rows with valid formats AND non-future dates
order_cleaning_df = order_cleaning_df.filter(
    F.try_to_date(F.col("order_date"), "yyyy-MM-dd").isNotNull() & 
    (F.try_to_date(F.col("order_date"), "yyyy-MM-dd") <= F.current_date())
)

display(order_cleaning_df)

In [0]:
from pyspark.sql import functions as F
order_quantity_clean=order_cleaning_df.filter(F.col("quantity")<=0)
display(order_quantity_clean)
order_quantity_clean.write.format("delta").mode("append").saveAsTable("workspace.silver.order_id_exception")

In [0]:
order_cleaning_df=order_cleaning_df.subtract(order_quantity_clean)
display(order_cleaning_df)

In [0]:
from pyspark.sql import functions as F
order_cleaning_df=order_cleaning_df.withColumn("order_id", F.trim(F.col("order_id")))
order_cleaning_df=order_cleaning_df.withColumn("customer_id", F.trim(F.col("customer_id")))
order_cleaning_df=order_cleaning_df.withColumn("product_id", F.trim(F.col("product_id")))
order_cleaning_df=order_cleaning_df.withColumn("order_date", F.trim(F.col("order_date")))


display(order_cleaning_df)
order_cleaning_df.sort(F.col("order_id"))
display(order_cleaning_df)
order_cleaning_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.orders_silver")


In [0]:
%sql
select * from workspace.silver.orders_silver

Now we will load products from bronze and start cleaning 

In [0]:
%sql
select * from workspace.bronze.products_raw

In [0]:
from pyspark.sql import functions as F
product_cleaning_df=spark.table("workspace.bronze.products_raw")
display(product_cleaning_df)

In [0]:
#lets check complete duplicates
# from pyspark.sql import functions as F
# prd_duplicates=product_cleaning_df.groupBy("product_id").agg(F.count("*").alias("count")).filter(F.col("count")>1)
# display(prd_duplicates)
# display(product_cleaning_df.filter(F.col("product_id")=="P003"))
#removing complete duplicates
product_cleaning_df=product_cleaning_df.dropDuplicates()
display(product_cleaning_df)

In [0]:
from pyspark.sql import functions as F
prd_name_null=product_cleaning_df.filter(F.col("product_name").isNull())
display(prd_name_null)
prd_name_null.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.products_exception")

In [0]:
product_cleaning_df=product_cleaning_df.subtract(prd_name_null)
display(product_cleaning_df)

now we have null in category we will compare with old vales and fill the nul and if in future new category comes it should be null untill category ids fixed

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define window partitioned by product_name
window_product = Window.partitionBy("product_name")

# 2. Infer category from non-null category of the same product_name
product_cleaning_df = product_cleaning_df.withColumn(
    "category",
    F.coalesce(F.col("category"), F.max("category").over(window_product))
)

display(product_cleaning_df)

Now we will check the price if null we will compare and add if mnus we will move to exception table 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define window partitioned by product_name and category
window_prod_cat = Window.partitionBy("product_name", "category")

# 2. Fill missing price using existing price for the same product_name + category
product_cleaning_df = product_cleaning_df.withColumn(
    "price",
    F.coalesce(F.col("price"), F.max("price").over(window_prod_cat))
)

display(product_cleaning_df)

In [0]:
from pyspark.sql import functions as F
product_price_negative=product_cleaning_df.filter(F.col("price")<0)
display(product_price_negative)
product_price_negative.write.format("delta").mode("append").saveAsTable("workspace.silver.products_exception")

In [0]:
product_cleaning_df=product_cleaning_df.subtract(product_price_negative)
display(product_cleaning_df)

In [0]:
from pyspark.sql import functions as F

# 1. Trim columns on product_cleaning_df
product_cleaning_df = product_cleaning_df.withColumn("product_id", F.trim(F.col("product_id")))
product_cleaning_df = product_cleaning_df.withColumn("product_name", F.trim(F.col("product_name")))
product_cleaning_df = product_cleaning_df.withColumn("category", F.trim(F.col("category")))

# 2. Sort product_cleaning_df by product_id
product_cleaning_df = product_cleaning_df.sort(F.col("product_id"))

# 3. Display the cleaned DataFrame
display(product_cleaning_df)

# 4. Save to Silver products table
product_cleaning_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.products_silver")

In [0]:
%sql
select * from workspace.silver.products_silver

Now we will load loyalty from bronze and start cleaning 

In [0]:
%sql
select * from workspace.bronze.loyalty_raw

In [0]:
loyalty_cleaning_df=spark.table("workspace.bronze.loyalty_raw")
display(loyalty_cleaning_df)

In [0]:
#We will inspect the complete duplicate row and we will rmeove it 
from pyspark.sql import functions as F
loyalty_duplicates=loyalty_cleaning_df.groupBy("loyalty_id").agg(F.count("*").alias("count")).filter(F.col("count")>1)
display(loyalty_duplicates)


In [0]:
loyalty_cleaning_df=loyalty_cleaning_df.dropDuplicates()
display(loyalty_cleaning_df)

In [0]:
# we are idenfying the loyalty id as null and moving to exception table 
from pyspark.sql import functions as F
loyalty_null=loyalty_cleaning_df.filter(F.col("loyalty_id").isNull())
display(loyalty_null)
loyalty_null.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.loyalty_exception")
#we are identifying the null customer id and moving to exception table

loyalty_cust_null=loyalty_cleaning_df.filter(F.col("customer_id").isNull())
display(loyalty_cust_null)
loyalty_cust_null.write.format("delta").mode("append").saveAsTable("workspace.silver.loyalty_exception")

In [0]:
loyalty_cleaning_df=loyalty_cleaning_df.subtract(loyalty_null)
display(loyalty_cleaning_df)

In [0]:
loyalty_cleaning_df=loyalty_cleaning_df.subtract(loyalty_cust_null)
display(loyalty_cleaning_df)

In [0]:
# we will look in to rewards it should not be null and negative number 
from pyspark.sql import functions as F
loyalty_reward_neg = loyalty_cleaning_df.filter(F.col("reward_points")<0)
display(loyalty_reward_neg)
loyalty_reward_neg.write.format("delta").mode("append").saveAsTable("workspace.silver.loyalty_exception")

In [0]:
from pyspark.sql import functions as F
loyalty_cleaning_df=loyalty_cleaning_df.subtract(loyalty_reward_neg)
# here we are making the rewards point null to zero
loyalty_cleaning_df = loyalty_cleaning_df.withColumn(
    "reward_points", F.coalesce(F.col("reward_points"), F.lit(0))
)

display(loyalty_cleaning_df)

In [0]:
from pyspark.sql import functions as F

# 1. Safely parse last_updated_date (invalid strings like 'not-a-date' become null)
loyalty_temp = loyalty_cleaning_df.withColumn(
    "last_updated_date_parsed",
    F.try_to_date(F.col("last_updated_date"), "yyyy-MM-dd")
)

# 2. Extract invalid date records (null/unparseable OR future dates) for exception handling
loyalty_date_exception = loyalty_temp.filter(
    F.col("last_updated_date_parsed").isNull() | 
    (F.col("last_updated_date_parsed") > F.current_date())
).drop("last_updated_date_parsed")

display(loyalty_date_exception)
loyalty_date_exception.write.format("delta").mode("append").saveAsTable("workspace.silver.loyalty_exception")

loyalty_cleaning_df=loyalty_cleaning_df.subtract(loyalty_date_exception)

In [0]:
display(loyalty_cleaning_df)

In [0]:
from pyspark.sql import functions as F

# 1. Parse last_updated_date safely
loyalty_temp = loyalty_cleaning_df.withColumn(
    "last_updated_date_parsed",
    F.try_to_date(F.col("last_updated_date"), "yyyy-MM-dd")
)

# 2. Extract records where last_updated_date is in the future
loyalty_last_date = loyalty_temp.filter(
    F.col("last_updated_date_parsed") > F.current_date()
).drop("last_updated_date_parsed")

display(loyalty_last_date)

In [0]:
%sql

select * from   workspace.silver.loyalty_exception

In [0]:
loyalty_cleaning_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.loyalty_silver")
display(loyalty_cleaning_df)